In [ ]:
from sentence_transformers import SentenceTransformer

# Load a small, fast embedding model we will use for semantic similarity.
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
# Example question and its embedding vector.
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)
v1

In [ ]:
# A candidate answer text and its embedding vector.
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)
dv

In [ ]:
# Dot product gives a quick similarity score between question and answer.
v1.dot(dv)

In [ ]:
# A different question, likely less related to the same answer.
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [ ]:
# Compare the second question against the same answer vector.
v2.dot(dv)

In [ ]:
from ingest import load_faq_data

# Load FAQ documents (question/answer pairs) from the ingest pipeline.
documents = load_faq_data()

In [ ]:
# Join question and answer into one text per document for embedding.
texts = []

for doc in documents:
    text = doc["question"] + "    " + doc["answer"]
    texts.append(text)

In [ ]:
# Quick preview to verify the combined text looks right.
texts[:5]

In [ ]:
# Progress bar helper for batch processing.
from tqdm.auto import tqdm

# Encode texts in batches to avoid big memory spikes.
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i: i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

In [ ]:
import numpy as np

# Convert list of vectors to a 2D NumPy array for fast math ops.
X = np.array(vectors)

In [ ]:
# Shape should be: (number_of_docs, embedding_dim).
X.shape

In [ ]:
# Encode the user query we want to search with.
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [ ]:
# Vectorized similarity: score every document against the query.
scores = X.dot(v_query)

In [ ]:
# Same scoring idea written explicitly with a loop.
scores = [v_query.dot(X[i]) for i in range(len(X))]

In [ ]:
# Index of the single best match and its score.
idx = np.argmax(scores)
idx, scores[idx]

In [ ]:
# Inspect the top document content.
documents[idx]

In [ ]:
# Get indices of the 5 highest scores (unsorted ascending slice).
top5 = np.argsort(scores)[-5:]

In [ ]:
# Reverse to show best-to-worst among top 5.
top5 = np.argsort(scores)[-5:]
top5 = top5[::-1]
top5

# Same result in one line.
topk = np.argsort(scores)[-5:][::-1]

In [ ]:
# Print each top match with its score for manual inspection.
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

In [ ]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [ ]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [ ]:
results[0]

In [ ]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [ ]:
results[0]

In [ ]:
from dotenv import load_dotenv
import os
from ollama import Client

load_dotenv()

client = Client(
    host="https://ollama.com",
    headers={'Authorization': 'Bearer ' + os.environ.get('OLLAMA_API_KEY')}
)

from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

from rag_helper import RAGBase

In [ ]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [ ]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=client,
)

In [ ]:
vector_assistant.rag("the program has already begun, can I still sign up?")

In [ ]:
!uv add sqlitesearch

In [32]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [33]:
vs_index.fit(vectors, documents)

In [34]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [38]:
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=2
)
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'}]

In [48]:
# When you're done with the index
vs_index.close()

In [40]:
# In a new Python session, you can reopen the index without re-computing embeddings:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [46]:
query_vector = model.encode("How do I run Kafka?")
results = vs_index.search(
    query_vector,
    num_results=2)

In [47]:
results

[{'id': '5ca6890c1a',
  'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: How to run producer/consumer/kstreams/etc in terminal',
  'answer': 'In the project directory, run:\n\n```bash\njava -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java\n```'},
 {'id': 'cd8a62fc55',
  'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: When running the producer/consumer/etc java scripts, no results retrieved or no message sent',
  'answer': 'For example, when running `JsonConsumer.java`, you might see:\n\n```\nConsuming form kafka started\n\nRESULTS:::0\n\nRESULTS:::0\n\nRESULTS:::0\n```\n\nOr when running `JsonProducer.java`, you might encounter:\n\n```\nException in thread "main" java.util.concurrent.ExecutionException: org.apache.kafka.common.errors.SaslAuthenticationException: Authentication failed\n```\n\n**Solution:**\n\n1. Ensure the `StreamsConfig.BO